In [1]:
WAREHOUSE = "analytics_warehouse"   
SCHEMA    = "gold"

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, FloatType, StringType, BitType

spark = SparkSession.builder.getOrCreate()

def write_to_warehouse(df, table: str, mode: str = "overwrite"):
    full_name = f"{WAREHOUSE}.{SCHEMA}.{table}"
    print(f"  Writing {df.count():,} rows → {full_name}  (mode={mode})")
    df.write.synapsesql(full_name, mode)
    print(f"  [OK] {full_name}")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, 4, Finished, Available, Finished, False)

ImportError: cannot import name 'BitType' from 'pyspark.sql.types' (/opt/spark/python/lib/pyspark.zip/pyspark/sql/types.py)

In [ ]:
dim_date_df = spark.table("dim_date")

# Warehouse column types must align exactly with the DDL
dim_date_cast = (
    dim_date_df
    .withColumn("date_key",     F.col("date_key").cast(IntegerType()))
    .withColumn("year",         F.col("year").cast("short"))
    .withColumn("quarter",      F.col("quarter").cast("short"))
    .withColumn("month",        F.col("month").cast("short"))
    .withColumn("week_of_year", F.col("week_of_year").cast("short"))
    .withColumn("day_of_month", F.col("day_of_month").cast("short"))
    .withColumn("day_of_week",  F.col("day_of_week").cast("short"))
    .withColumn("is_weekend",   F.col("is_weekend").cast(IntegerType()))    # BIT → INT
    .withColumn("is_leap_year", F.col("is_leap_year").cast(IntegerType()))
    .select(
        "date_key", "full_date", "year", "quarter", "month", "month_name",
        "month_abbr", "week_of_year", "day_of_month", "day_of_week",
        "day_name", "day_abbr", "is_weekend", "is_leap_year",
        "year_month", "year_quarter",
    )
)

write_to_warehouse(dim_date_cast, "DimDate")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:
dim_zone_df = (
    spark.table("dim_zone")
    .withColumn("location_id",  F.col("location_id").cast(IntegerType()))
    .withColumn("is_airport",   F.col("is_airport").cast(IntegerType()))
    .select("location_id", "borough", "zone_name", "service_zone", "is_airport")
)

write_to_warehouse(dim_zone_df, "DimZone")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:
dim_location_df = (
    spark.table("air_quality")
    .select(
        "location_key", "location_id", "location_name",
        "city", "country", "latitude", "longitude"
    )
    .dropDuplicates(["location_key"])
    .withColumn("location_id", F.col("location_id").cast(LongType()))
    .withColumn("latitude",    F.col("latitude").cast(FloatType()))
    .withColumn("longitude",   F.col("longitude").cast(FloatType()))
)

write_to_warehouse(dim_location_df, "DimLocation")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:

dim_fx_df = (
    spark.table("fx_daily")
    .withColumn("year", F.year("rate_date").cast(IntegerType()))
    .groupBy("year")
    .agg(
        F.round(F.avg("usd_eur_rate"), 6).alias("avg_usd_eur_rate"),
        F.round(F.min("usd_eur_rate"), 6).alias("min_usd_eur_rate"),
        F.round(F.max("usd_eur_rate"), 6).alias("max_usd_eur_rate"),
        F.count("*").cast(IntegerType()).alias("trading_days"),
    )
    .orderBy("year")
)


write_to_warehouse(dim_fx_df, "DimFX")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:
dim_gdp_df = (
    spark.table("gdp_fx")
    .select(
        "country_iso3", "country_name", "year",
        "gdp_usd", "avg_usd_eur_rate", "gdp_eur"
    )
    .withColumn("year",             F.col("year").cast(IntegerType()))
    .withColumn("gdp_usd",          F.col("gdp_usd").cast(FloatType()))
    .withColumn("avg_usd_eur_rate", F.col("avg_usd_eur_rate").cast(FloatType()))
    .withColumn("gdp_eur",          F.col("gdp_eur").cast(FloatType()))
    .dropDuplicates(["country_iso3", "year"])
    .orderBy("country_iso3", "year")
)

write_to_warehouse(dim_gdp_df, "DimGDP")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:
print("\n--- Row counts in analytics_warehouse ---")
for table in ["DimDate", "DimZone", "DimLocation", "DimFX", "DimGDP"]:
    full = f"{WAREHOUSE}.{SCHEMA}.{table}"
    cnt  = spark.read.synapsesql(full).count()
    print(f"  {table:20s} : {cnt:,}")

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)

In [ ]:
# =============================================================================
# Phase 3 – Gold: Load Dimension Tables → analytics_warehouse
#
# Source  : silver Lakehouse (Default on this notebook)
#           silver.dim_date, silver.dim_zone, silver.air_quality,
#           silver.gdp_fx, silver.fx_daily
# Target  : analytics_warehouse  (gold schema)
#           gold.DimDate, gold.DimZone, gold.DimLocation,
#           gold.DimFX, gold.DimGDP
#

StatementMeta(, e7443567-f42c-47aa-b411-3bcb1f39ef00, -1, Cancelled, , Cancelled, True)